# Demo 1 — Unpaired t-test equivalent

This demo uses the classic `sleep` dataset (R base; Cushny & Peebles 1905, later
made famous by Student 1908), which records the extra hours of sleep gained by 10
patients under two different soporific drugs. Treating the two drug groups as
independent, it asks whether the average sleep gain differs between them.

The model is a plain linear model with no random effects:

    extra ~ group

With a two-level factor and effects coding this is algebraically identical to an
independent-samples (Student) t-test — the ANOVA F equals t², the degrees of
freedom are n − 2, and the p-value matches exactly — so it reproduces the
classical result while using the same machinery as every other demo.

## Setup

In [ ]:
import os
try:
    notebook_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    notebook_dir = os.getcwd()  # JupyterLab sets CWD to the notebook directory
os.chdir(notebook_dir)

import matplotlib.pyplot as plt
from kbstatpy import Kbstat, KbstatOptions

## Options

The `sleep` dataset has two groups (`1` and `2`), renamed here to their drug labels.
No random effects (`id` left empty) — this is a plain between-subjects design.


In [ ]:
options = KbstatOptions()
options.in_file  = os.path.join(notebook_dir, '../data/sleep.csv')
options.out_dir  = ''   # empty: show results inline only; set a folder to also save them
options.y        = 'extra'
options.y_units  = 'h'
options.x        = 'group'
options.rename   = 'extra -> ExtraSleep; group -> DrugGroup'

## Model fitting

`distribution = 'normal'` (default) fits a Gaussian linear model via `lm()` in R.


In [ ]:
kb = Kbstat(options)
kb.fit()
print(f'Formula : {kb._build_formula()}')
print(f'AIC     : {kb.AIC:.3f}')
print(f'BIC     : {kb.BIC:.3f}')
print(f'logLik  : {kb.logLik:.3f}')


## ANOVA table (Type III)

For a two-level factor without random effects, F = t² and the p-value is identical
to an independent-samples t-test — see [STATISTICAL_NOTES.md](../STATISTICAL_NOTES.md).


In [ ]:
kb.anova()
kb.anova_table


## Post-hoc pairwise comparisons

With only two levels there is one comparison. Holm correction has no effect here.


In [ ]:
kb.posthoc()
kb.posthoc_table[['group_1', 'group_2', 'emm_1', 'emm_2', 'diff', 't', 'df', 'pCorr', 'significance']]

## Data plot

Violin + jitter scatter with the model 95 % CI bar and EMM dot overlaid.
The EMM equals the raw group mean here — a plain LM with no covariates or random effects.

> An interactive version with hover tooltips is written to `DataPlots.html` when `out_dir` is set.


In [ ]:
kb.plot_data()
plt.show()


## Diagnostic plots

Six panels checking model assumptions. For a two-group LM on 20 observations
expect some noise — look for gross violations rather than perfection.

See [STATISTICAL_NOTES.md](../STATISTICAL_NOTES.md#diagnostic-plots) for panel-by-panel interpretation.


In [ ]:
kb.plot_diagnostics()
plt.show()


## Save results

With `out_dir` set, `kb.save()` writes all output files there (empty here, so this only shows a notice):
`Anova.xlsx`, `Posthoc.xlsx`, `Statistics.xlsx`, `DataPlots.pdf/.png/.html`,
`Diagnostics.pdf/.png/.html`, `Data.csv`, `Summary.txt`.


In [ ]:
kb.save()


## Interpretation

- The ANOVA table shows whether DrugGroup has a significant effect on ExtraSleep.
- The post-hoc table gives the estimated mean difference with 95 % CI and corrected p-value.
- For a two-group LM the result is numerically identical to an independent-samples t-test
  (same F = t², same df = n − 2, same p-value) regardless of whether the groups are balanced.
